# Mixup-GRPO 优化实验 v2 — TinyLLaVA-Video-R1（Colab）

本 Notebook 用于 **重启阶段** 的 GRPO/CPPO 对比实验，目标是在 A100 上放大 `num_generations`（默认 **8**），使 CPPO 剪枝带来更明显的训练加速，并与 v1 结果区分。

**与 v1（`GRPO测试优化-基于tiny video R1的GRPO优化.ipynb`）的主要区别：**
- `num_generations = 8`（v1 为 4）
- `max_steps = 100`（v1 为 50，减小计时噪声）
- 输出目录：`grpo_A_g8` / `grpo_CPPO_g8_p50` / `grpo_CPPO_g8_p75`
- 详细规划见同目录 `schedule.md`

**建议顺序：**
1. 环境与路径检查 → 路径配置 → 准备数据集 → 安装依赖
2. Trainer 配置（g8 + 重置 CPPO）
3. 策略 A2（GRPO 基线）→ CPPO 补丁 → 策略 C2 / C3
4. 运行结果对比汇总


---
## ⚠️ 每次重新连接 Colab 后必做

| 顺序 | 单元 | 说明 |
|------|------|------|
| 1 | **挂载 Drive** | `drive.mount('/content/drive')` |
| 2 | **环境与路径检查** | 确认 GPU 是否为 A100-80GB |
| 3 | **路径配置** | 设置 `REPO`、`OUT_BASE`、`SMALL_JSONL`、`CKPT` |
| 4 | **准备小规模数据集** | 生成 `nextqa_small50.jsonl` |
| 5 | **解压/确认 REPO** | 确认 TinyLLaVA-Video-R1 存在 |
| 6 | **安装依赖** | pip install -e .、trl、deepspeed 等 |
| 7 | **Trainer 配置（g8）** | `num_generations=8`，重置 `cppo_pruning_rate=0` |
| 8 | **训练与对比** | A2 → C2 → C3 → 结果汇总 |

若 GPU 降级为 T4，请将路径配置中的 `NUM_GENERATIONS` 改回 `4`。


---
## 1. 环境与路径检查

运行后会打印：Python 版本、是否在 Colab、GPU 型号与显存、云盘路径是否存在。请根据输出在下一节确认或修改路径变量。


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# 环境与路径检查（无需修改，直接运行）
import sys
import os

def check_path(p, name):
    exists = os.path.exists(p)
    print(f"  [{name}] {p}  ->  {'存在' if exists else '不存在'}")
    if exists and os.path.isdir(p):
        try:
            print(f"        子项(前5个): {os.listdir(p)[:5]}")
        except Exception as e:
            print(f"        listdir 失败: {e}")
    return exists

print("=== Python ===")
print(sys.version)
print("\n=== 是否 Colab ===")
try:
    import google.colab
    print("是 Colab")
    IN_COLAB = True
except ImportError:
    print("否（本地 Jupyter）")
    IN_COLAB = False

print("\n=== GPU ===")
GPU_OK = False
try:
    import torch
    print(f"PyTorch: {torch.__version__}, CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  设备: {name}, 显存(GB): {mem:.2f}")
        GPU_OK = "A100" in name and mem > 70
        if GPU_OK:
            print("  [建议] A100 高配，可使用 NUM_GENERATIONS=8")
        else:
            print("  [注意] 非 A100 或大显存卡，建议将 NUM_GENERATIONS 降为 4")
except Exception as e:
    print(f"  {e}")

print("\n=== 常见云盘路径检查 ===")
candidates = [
    "/content/drive/MyDrive/MixUpLLaVA-video-r1",
    "/content/drive/MyDrive/MixUpLLaVA-video-r1/repo",
    "/content/drive/MyDrive/MixUpLLaVA-video-r1/data/dataset",
    "/content/drive/MyDrive/MixUpLLaVA-video-r1/checkpoints/coldstart",
]
for p in candidates:
    check_path(p, "dir")

base = "/content/drive/MyDrive/MixUpLLaVA-video-r1/repo"
if os.path.exists(base):
    for name in os.listdir(base):
        sub = os.path.join(base, name)
        if os.path.isdir(sub):
            check_path(sub, name)


=== Python ===
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

=== 是否 Colab ===
是 Colab

=== GPU ===
PyTorch: 2.11.0+cu128, CUDA 可用: True
  设备: NVIDIA A100-SXM4-80GB, 显存(GB): 85.09
  [建议] A100 高配，可使用 NUM_GENERATIONS=8

=== 常见云盘路径检查 ===
  [dir] /content/drive/MyDrive/tiny-video-r1-GRPO  ->  存在
        子项(前5个): ['repo', 'data', 'outputs', 'checkpoints']
  [dir] /content/drive/MyDrive/tiny-video-r1-GRPO/repo  ->  存在
        子项(前5个): ['TinyLLaVA-Video-R1', 'TinyLLaVA-Video-R1-CPPO']
  [dir] /content/drive/MyDrive/tiny-video-r1-GRPO/data/dataset  ->  存在
        子项(前5个): ['.cache', '.gitattributes', 'README.md', 'nextqa-coldstart-16.json', 'nextqa_0-30s.jsonl']
  [dir] /content/drive/MyDrive/tiny-video-r1-GRPO/checkpoints/coldstart  ->  存在
        子项(前5个): ['.cache', 'merges.txt', 'generation_config.json', 'README.md', '.gitattributes']
  [TinyLLaVA-Video-R1] /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1  ->  存在
        子项(前5个): ['.git', '.gitignore', 'LICENSE', 'READ

---
## 2. 路径配置

根据上一步检查结果修改 `PROJECT_DIR`（若云盘路径不同）。本轮实验关键超参也在此定义。


In [ ]:
# ========== 路径配置（请根据「环境与路径检查」结果修改） ==========
import os

PROJECT_DIR = "/content/drive/MyDrive/MixUpLLaVA-video-r1"
REPO_NAME = "TinyLLaVA-Video-R1"
REPO = os.path.join(PROJECT_DIR, "repo", REPO_NAME.replace(".zip", ""))

DATA_ROOT = os.path.join(PROJECT_DIR, "data", "dataset").rstrip("/") + "/"
DATA_ROOT_STRIP = os.path.join(PROJECT_DIR, "data", "dataset")
DATA_JSONL = os.path.join(DATA_ROOT_STRIP, "nextqa_0-30s_10p_seed42.jsonl") if os.path.exists(os.path.join(DATA_ROOT_STRIP, "nextqa_0-30s_10p_seed42.jsonl")) else os.path.join(DATA_ROOT_STRIP, "nextqa_0-30s.jsonl")
CKPT = os.path.join(PROJECT_DIR, "checkpoints", "coldstart")
OUT_BASE = os.path.join(PROJECT_DIR, "outputs")

# v2 实验超参（见 schedule.md）
SMALL_N = 50
SMALL_JSONL = os.path.join(DATA_ROOT_STRIP, f"nextqa_small{SMALL_N}.jsonl")
NUM_GENERATIONS = 8      # A100 推荐 8；T4 请改为 4
MAX_STEPS = 100          # 时间不够可改为 50
NUM_FRAMES = 2
NUM_QUERIES = 32
MODEL_MAX_LENGTH = 256

# 输出目录命名
OUT_A2 = os.path.join(OUT_BASE, "grpo_A_g8")
OUT_C2 = os.path.join(OUT_BASE, "grpo_CPPO_g8_p50")
OUT_C3 = os.path.join(OUT_BASE, "grpo_CPPO_g8_p75")

print("REPO:", REPO, "->", os.path.exists(REPO))
print("DATA_JSONL:", DATA_JSONL, "->", os.path.exists(DATA_JSONL))
print("CKPT:", CKPT)
print("OUT_BASE:", OUT_BASE)
print("SMALL_JSONL:", SMALL_JSONL)
print("NUM_GENERATIONS:", NUM_GENERATIONS, "| MAX_STEPS:", MAX_STEPS)
print("OUT_A2:", OUT_A2)
print("OUT_C2:", OUT_C2)
print("OUT_C3:", OUT_C3)


---
## 3. 准备小规模数据集

从完整 NextQA 子集截取前 `SMALL_N` 条写入 `SMALL_JSONL`。


In [ ]:
import itertools

if not os.path.exists(DATA_JSONL):
    raise FileNotFoundError(f"请先准备数据: {DATA_JSONL}")

with open(DATA_JSONL, "r", encoding="utf-8") as rf, open(SMALL_JSONL, "w", encoding="utf-8") as wf:
    for rec in itertools.islice(rf, SMALL_N):
        wf.write(rec)
print(f"已写入 {SMALL_N} 条到 {SMALL_JSONL}")


---
## 4. 安装依赖与准备 Repo

若 repo 为 zip 会先解压；然后在 REPO 目录执行 `pip install -e .` 及训练依赖。


In [ ]:
# 若 repo 为 zip 则解压；否则确认 REPO 已存在即可
repo_parent = os.path.join(PROJECT_DIR, "repo")
zip_path = os.path.join(repo_parent, "TinyLLaVA-Video-R1-main.zip")
extracted = os.path.join(repo_parent, "TinyLLaVA-Video-R1-main")
REPO = os.path.join(repo_parent, REPO_NAME.replace(".zip", ""))

if os.path.exists(REPO):
    print("REPO 已存在，无需解压:", REPO)
elif not os.path.exists(extracted) and os.path.exists(zip_path):
    import zipfile
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(repo_parent)
    print("已解压到", extracted)
elif os.path.exists(extracted):
    print("Repo 已存在:", extracted)
else:
    print("未找到 zip 或已解压目录，请确认 REPO 路径:", REPO)


In [ ]:
# 确认当前 REPO
REPO = os.path.join(repo_parent, REPO_NAME.replace(".zip", ""))
if not os.path.exists(REPO):
    REPO = os.path.join(repo_parent, "TinyLLaVA-Video-R1-main")
assert os.path.exists(REPO), f"REPO 不存在: {REPO}"
print("当前 REPO:", REPO)


In [ ]:
# 安装依赖（在 REPO 目录下执行 pip install -e .）
import subprocess
import sys

cmds = [
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
    [sys.executable, "-m", "pip", "install", "-q", "trl", "datasets", "pytorchvideo", "decord", "transformers", "accelerate", "deepspeed"],
]
for cmd in cmds:
    subprocess.run(cmd, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=REPO, check=True)

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "math_verify"], check=True)
except Exception as e:
    print("math_verify 安装失败:", e)

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-attn==2.7.3", "--no-build-isolation"], check=True)
    print("flash-attn 安装成功")
except Exception as e:
    print("flash-attn 未安装，训练时将使用 eager attention:", e)


---
## 5. Trainer 配置（v2：num_generations=8）

将 `tinyllava_trainer_reason.py` 设为：
- `num_generations = NUM_GENERATIONS`（默认 8）
- `cppo_pruning_rate = 0`（GRPO 基线，跑 CPPO 前再单独打补丁）
- `num_frame = 8`（与 v1 显存减负一致，可按显存调整）

**跑 CPPO 实验前请从云盘恢复策略 A 版 trainer，或重新运行本单元。**


In [ ]:
# 配置 trainer：g8 + 关闭 CPPO
trainer_path = os.path.join(REPO, "tinyllava", "train", "tinyllava_trainer_reason.py")
with open(trainer_path, "r", encoding="utf-8") as f:
    content = f.read()

import re
# num_generations
for pat, val in [
    (r"self\.num_generations = \d+.*", f"self.num_generations = {NUM_GENERATIONS}  # v2 experiment"),
    (r"self\.num_frame = \d+.*", "self.num_frame = 8"),
]:
    if re.search(pat, content):
        content = re.sub(pat, val, content, count=1)

# 重置 CPPO 为关闭（支持多种写法）
replacements = [
    ("self.cppo_pruning_rate = 0.5  # CPPO: 50% completion pruning", "self.cppo_pruning_rate = 0.0  # GRPO baseline"),
    ("self.cppo_pruning_rate = 0.75  # CPPO: 75% completion pruning", "self.cppo_pruning_rate = 0.0  # GRPO baseline"),
    ("self.cppo_pruning_rate = getattr(args, "cppo_pruning_rate", 0.0)", "self.cppo_pruning_rate = 0.0  # GRPO baseline"),
]
for old, new in replacements:
    if old in content:
        content = content.replace(old, new)

with open(trainer_path, "w", encoding="utf-8") as f:
    f.write(content)
print(f"已配置 trainer: num_generations={NUM_GENERATIONS}, num_frame=8, cppo_pruning_rate=0.0")
print("路径:", trainer_path)


---
## 6. 训练工具函数（ZeRO-3 + 统一命令）

以下单元定义 DeepSpeed 配置与训练启动函数，供 A2/C2/C3 复用，保证超参一致。


In [ ]:
import os
import json
import subprocess

def ensure_zero3_config(repo):
    ds_scripts = os.path.join(repo, "scripts")
    zero3_offload_path = os.path.join(ds_scripts, "zero3_offload.json")
    os.makedirs(ds_scripts, exist_ok=True)
    with open(zero3_offload_path, "w") as f:
        json.dump({
            "fp16": {"enabled": "auto", "loss_scale": 0, "loss_scale_window": 1000, "initial_scale_power": 16, "hysteresis": 2, "min_loss_scale": 1},
            "bf16": {"enabled": "auto"},
            "zero_optimization": {
                "stage": 3,
                "offload_optimizer": {"device": "none", "pin_memory": True},
                "offload_param": {"device": "cpu", "pin_memory": True},
                "overlap_comm": True, "contiguous_gradients": True,
                "sub_group_size": 1000000000, "reduce_bucket_size": "auto",
                "stage3_prefetch_bucket_size": "auto", "stage3_param_persistence_threshold": "auto",
                "stage3_max_live_parameters": 1000000000, "stage3_max_reuse_distance": 1000000000,
                "stage3_gather_16bit_weights_on_model_save": True,
            },
            "gradient_accumulation_steps": "auto", "gradient_clipping": "auto",
            "steps_per_print": 100, "train_batch_size": "auto", "train_micro_batch_size_per_gpu": "auto",
            "wall_clock_breakdown": False,
        }, f, indent=2)
    return zero3_offload_path

def build_train_cmd(output_dir, run_name, zero3_path, attn="flash_attention_2"):
    return [
        "deepspeed", "--num_gpus=1", os.path.join(REPO, "tinyllava/train/train.py"),
        "--deepspeed", zero3_path,
        "--video_data_path", DATA_ROOT, "--video_folder", SMALL_JSONL,
        "--is_multimodal", "True", "--conv_version", "qwen2_base",
        "--model_name_or_path", "Qwen/Qwen2.5-3B",
        "--vision_tower", "google/siglip-so400m-patch14-384",
        "--connector_type", "groupresampler",
        "--num_frames", str(NUM_FRAMES), "--num_queries", str(NUM_QUERIES),
        "--mm_vision_select_layer", "-2", "--image_aspect_ratio", "square",
        "--attn_implementation", attn, "--bf16", "True",
        "--training_recipe", "common", "--tune_type_llm", "full",
        "--tune_type_vision_tower", "frozen", "--tune_vision_tower_from_layer", "0",
        "--tune_type_connector", "full", "--group_by_modality_length", "False",
        "--pretrained_model_path", CKPT,
        "--output_dir", output_dir,
        "--num_train_epochs", "1", "--max_steps", str(MAX_STEPS),
        "--per_device_train_batch_size", "1", "--gradient_accumulation_steps", "1",
        "--evaluation_strategy", "no", "--save_strategy", "no", "--report_to", "none",
        "--learning_rate", "5e-6", "--weight_decay", "0.0", "--warmup_ratio", "0.03",
        "--lr_scheduler_type", "cosine", "--logging_steps", "5", "--tf32", "False",
        "--model_max_length", str(MODEL_MAX_LENGTH), "--gradient_checkpointing", "True",
        "--dataloader_num_workers", "2", "--lazy_preprocess", "True", "--tokenizer_use_fast", "False",
        "--run_name", run_name,
    ]

def run_training(output_dir, run_name):
    os.makedirs(OUT_BASE, exist_ok=True)
    zero3_path = ensure_zero3_config(REPO)
    cmd = build_train_cmd(output_dir, run_name, zero3_path)
    env = os.environ.copy()
    env["PYTHONPATH"] = REPO + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    print("执行训练:", run_name, "->", output_dir)
    p = subprocess.run(cmd, cwd=REPO, env=env, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.returncode != 0 and p.stderr:
        print("=== stderr ===")
        print(p.stderr)
    if p.returncode != 0:
        raise SystemExit(p.returncode)
    print("训练完成:", output_dir)

def apply_cppo_patch(pruning_rate):
    trainer_path = os.path.join(REPO, "tinyllava", "train", "tinyllava_trainer_reason.py")
    with open(trainer_path, "r", encoding="utf-8") as f:
        content = f.read()
    import re
    new_line = f"self.cppo_pruning_rate = {pruning_rate}  # CPPO pruning"
    if re.search(r"self\.cppo_pruning_rate = .*", content):
        content = re.sub(r"self\.cppo_pruning_rate = .*", new_line, content, count=1)
    else:
        raise RuntimeError("trainer 中未找到 cppo_pruning_rate，请确认已使用含 CPPO 的 tinyllava_trainer_reason.py")
    with open(trainer_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"已设置 cppo_pruning_rate={pruning_rate}")

print("训练工具函数已就绪（ensure_zero3_config / build_train_cmd / run_training / apply_cppo_patch）")


---
## 7. 策略 A2：GRPO Baseline（g8，无剪枝）

与 CPPO 对比的公平基线。运行前请确认已执行「Trainer 配置（g8）」且 `cppo_pruning_rate=0`。


In [ ]:
try:
    _ = REPO, OUT_BASE, DATA_ROOT, SMALL_JSONL, CKPT, OUT_A2, run_training
except NameError as e:
    raise RuntimeError("环境未就绪：请按顺序运行前置单元（含「训练工具函数」）") from e

run_training(OUT_A2, "grpo_A_g8")


---
## 8. 策略 C2：CPPO（g8，pruning_rate=0.5）

剪枝 50% 低 |advantage| completion。跑前会自动打 CPPO 补丁。


In [ ]:
apply_cppo_patch(0.5)
run_training(OUT_C2, "grpo_CPPO_g8_p50")


---
## 9. 策略 C3：CPPO（g8，pruning_rate=0.75）（可选）

更激进剪枝，观察加速上限与 reward 是否下降。显存/时间允许时再跑。


In [ ]:
apply_cppo_patch(0.75)
run_training(OUT_C3, "grpo_CPPO_g8_p75")


---
## 10. 结果对比：A2 vs C2 vs C3

读取各策略 `trainer_state.json`，汇总 reward、loss、train_runtime，并计算相对 A2 的加速比。


In [ ]:
import os
import json

def _read_last_metrics(out_dir):
    path = os.path.join(out_dir, "trainer_state.json")
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        state = json.load(f)
    log = state.get("log_history", [])
    last_with_loss, last_runtime = None, None
    for e in reversed(log):
        if last_with_loss is None and ("loss" in e or "reward" in e):
            last_with_loss = e
        if last_runtime is None and "train_runtime" in e:
            last_runtime = e
        if last_with_loss and last_runtime:
            break
    return {"metrics": last_with_loss, "runtime": last_runtime}

try:
    _ = OUT_A2, OUT_C2, OUT_C3
except NameError:
    OUT_BASE = OUT_BASE if "OUT_BASE" in dir() else "/content/drive/MyDrive/MixUpLLaVA-video-r1/outputs"
    OUT_A2 = os.path.join(OUT_BASE, "grpo_A_g8")
    OUT_C2 = os.path.join(OUT_BASE, "grpo_CPPO_g8_p50")
    OUT_C3 = os.path.join(OUT_BASE, "grpo_CPPO_g8_p75")

rows = []
baseline_rt = None
for name, d in [
    ("A2 GRPO baseline (g8)", OUT_A2),
    ("C2 CPPO p=0.5 (g8)", OUT_C2),
    ("C3 CPPO p=0.75 (g8)", OUT_C3),
]:
    data = _read_last_metrics(d)
    print(f"--- {name} ---")
    print("  目录:", d, "| 存在:", os.path.isdir(d))
    if not data:
        print("  无 trainer_state.json")
        rows.append({"name": name, "dir": d})
        continue
    m, r = data["metrics"], data["runtime"]
    rt = r.get("train_runtime") if r else None
    if name.startswith("A2") and rt:
        baseline_rt = rt
    row = {
        "name": name,
        "step": m.get("step") if m else None,
        "loss": m.get("loss") if m else None,
        "reward": m.get("reward") if m else None,
        "reward_std": m.get("reward_std") if m else None,
        "train_runtime": rt,
        "steps_per_second": r.get("train_steps_per_second") if r else None,
    }
    if baseline_rt and rt and not name.startswith("A2"):
        row["speedup_vs_A2_%"] = round((baseline_rt - rt) / baseline_rt * 100, 2)
    rows.append(row)
    if m:
        print(f"  step={m.get('step')} loss={m.get('loss')} reward={m.get('reward')}")
    if r:
        print(f"  runtime={rt}s steps/s={r.get('train_steps_per_second')}")
        if "speedup_vs_A2_%" in row:
            print(f"  相对 A2 加速: {row['speedup_vs_A2_%']}%")

print("\n=== 汇总表 ===")
for row in rows:
    print(row)


---
## 附录

- **v1 结果**：`outputs/grpo_A_baseline_small`、`grpo_CPPO_small`（num_generations=4，CPPO 加速约 0.7%）
- **flash_attention**：安装失败时在 `build_train_cmd` 中把 `attn` 改为 `eager`
- **OOM**：将 `NUM_GENERATIONS` 降为 4，或减小 `MAX_STEPS` / `NUM_FRAMES`
- **公平对比**：A2 与 C2/C3 仅差 `cppo_pruning_rate` 与输出目录；不要叠加策略 B 的 reward 修改
- 规划文档：同目录 `schedule.md`
